In [ ]:
# les données climatiques sont volumineuses (un fichier par département)
# pour ne pas les télécharger à la main, on crée un programme pour les importer automatiquement 
# en sélectionnant uniquement les variables d'intérêt

import pandas as pd
import numpy as np

# pour chaque département, on va procéder de la même façon
# on crée donc une fonction qui prend le département comme argument

def agreg_dpt(DEP):
    url = "https://object.files.data.gouv.fr/meteofrance/data/synchro_ftp/BASE/MENS/MENSQ_" + DEP + "_previous-1950-2023.csv.gz"
    df=pd.read_csv(url, sep=";")
    df_filtre = df[['NOM_USUEL', "AAAAMM", "TX", "TXMIN", "NBJTX0", "NBJTX25", "NBJTX30", "NBJTX35","NBJNEIG", "NBJSOLNG"]]
# on crée une variable ne contenant que l'annee 
    df_filtre['AAAA'] = df_filtre['AAAAMM'].astype(str).str[:4].astype(int)
    df_filtre['MM']= df_filtre['AAAAMM'].astype(str).str[5:6].astype(int)
# on sélectionne nos mois et années d'intérêt
    df_filtre = df_filtre[df_filtre.AAAA.isin(list(range(2011, 2022)))]
    df_filtre = df_filtre[df_filtre.MM.isin([1,2,3,6,7,8,9,12])]
# on calcule la moyenne départementale pour toutes les variables
    df_filtre['TX_num']=df_filtre.TX.astype("float64")
    df_filtre['TXMIN_num']=df_filtre.TXMIN.astype("float64")
    df_filtre = df_filtre.groupby(['AAAA','MM'])[["TX_num", "TXMIN_num", "NBJTX0", "NBJTX25", "NBJTX30", "NBJTX35","NBJNEIG", "NBJSOLNG"]].mean()
    df_filtre['DEP'] = DEP
    return(df_filtre)

# on prépare une boucle pour importer un à un les fichiers départements
# et concaténer les outputs de la fonction agreg_dpt
liste_dep = np.arange(2,96)
df = agreg_dpt("01")
for i in liste_dep:
    num = f'{i:02}'
    df = pd.concat([df, agreg_dpt(num)])

/var/folders/wk/h4llx_bj1yz15v8hrjw1vp8r0000gn/T/ipykernel_39070/237133493.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtre['AAAA'] = df_filtre['AAAAMM'].astype(str).str[:4].astype(int)
/var/folders/wk/h4llx_bj1yz15v8hrjw1vp8r0000gn/T/ipykernel_39070/237133493.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtre['MM']= df_filtre['AAAAMM'].astype(str).str[5:6].astype(int)
/var/folders/wk/h4llx_bj1yz15v8hrjw1vp8r0000gn/T/ipykernel_39070/237133493.py:15: SettingWithCopyWarning: 
A value

In [ ]:
# on remet année et mois (devenues index) en variables normales
df = df.reset_index()

In [ ]:
# saison (été ou hiver)
conditions1 = [
    (df['MM'] <= 3) | (df['MM'] == 12),
    (df['MM'] >= 6) & (df['MM'] <= 9)
    ]

values1 = ['hiver', 'été']

df['saison'] = np.select(conditions1, values1, default='Other')

# période (avant ou après 2015)
conditions2 = [
    (df['AAAA'] <= 2015),
    (df['AAAA'] > 2015)
    ]

values2 = ['avant_2015', 'apres_2015']

df['periode'] = np.select(conditions2, values2, default='Other')



In [132]:
# pour tester : évolution moyenne l'été dans les bouches du rhone entre avant 2015 vs apres 2015

df[df['DEP']=="13"].groupby(['saison', 'periode'])['NBJTX30'].mean()

saison  periode   
hiver   apres_2015     0.000000
        avant_2015     0.000000
été     apres_2015    15.372454
        avant_2015    11.300184
Name: NBJTX30, dtype: float64